Task C) Design and implement a CNN model (with 4+ layers of convolutions) to classify multi category image datasets. Use the concept of regularization and dropout while designing the CNN model. Use the Fashion MNIST datasets. Record the Training accuracy and Test accuracy
corresponding to the following architectures: a. Base Model b. Model with L1 Regularization c. Model with L2 Regularization d. Model with Dropout e. Model with both L2 (or L1) and Dropout

In [1]:
# Simple CNN with 5 conv layers for Fashion MNIST

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Set random seed for reproducibility (student style)
tf.random.set_seed(42)

In [2]:
# --------------------- Load and prepare data ---------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Add channel dimension for CNN (28,28,1)
x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]

# One-hot encode labels (10 classes)
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# --------------------- Model creation function ---------------------
def create_cnn(reg_type=None, dropout_rate=0.0):
    """
    Build a CNN with 5 convolutional layers.
    reg_type: 'l1', 'l2', or None (no weight regularization)
    dropout_rate: 0.0 means no Dropout layer
    """
    model = keras.Sequential()

    # ---- Choose regularizer if any ----
    if reg_type == 'l1':
        reg = regularizers.l1(0.001)
    elif reg_type == 'l2':
        reg = regularizers.l2(0.001)
    else:
        reg = None

    # ---- Conv Block 1 (conv + pool) ----
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same',
                            kernel_regularizer=reg, input_shape=(28,28,1)))
    model.add(layers.MaxPooling2D((2,2)))   # 14x14

    # ---- Conv Block 2 (conv + pool) ----
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same',
                            kernel_regularizer=reg))
    model.add(layers.MaxPooling2D((2,2)))   # 7x7

    # ---- Conv Block 3 (conv only, no pool) ----
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same',
                            kernel_regularizer=reg))   # 7x7

    # ---- Conv Block 4 (conv + pool) ----
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same',
                            kernel_regularizer=reg))
    model.add(layers.MaxPooling2D((2,2)))   # 3x3

    # ---- Conv Block 5 (conv only, no pool) ----
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same',
                            kernel_regularizer=reg))   # 3x3

    # ---- Classifier head ----
    model.add(layers.Flatten())
    if dropout_rate > 0:
        model.add(layers.Dropout(dropout_rate))
    # Output layer (10 categories)
    model.add(layers.Dense(10, activation='softmax', kernel_regularizer=reg))

    return model


In [4]:
# --------------------- Train & evaluate helper ---------------------
def train_and_evaluate(model, model_name):
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    print(f"\n--- Training {model_name} ---")
    history = model.fit(x_train, y_train,
                        epochs=10,
                        batch_size=64,
                        validation_data=(x_test, y_test),
                        verbose=1)  # shows progress bars
    train_acc = history.history['accuracy'][-1]
    test_acc = history.history['val_accuracy'][-1]
    print(f"{model_name} -> Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    return train_acc, test_acc

In [5]:
# --------------------- Run all architectures ---------------------
results = {}

# (a) Base Model – no regularization, no dropout
model_base = create_cnn(reg_type=None, dropout_rate=0.0)
results['Base'] = train_and_evaluate(model_base, "Base Model")

# (b) Model with L1 Regularization
model_l1 = create_cnn(reg_type='l1', dropout_rate=0.0)
results['L1'] = train_and_evaluate(model_l1, "L1 Regularization")

# (c) Model with L2 Regularization
model_l2 = create_cnn(reg_type='l2', dropout_rate=0.0)
results['L2'] = train_and_evaluate(model_l2, "L2 Regularization")

# (d) Model with Dropout
model_drop = create_cnn(reg_type=None, dropout_rate=0.5)
results['Dropout'] = train_and_evaluate(model_drop, "Dropout")

# (e) Model with L2 Regularization + Dropout
model_l2_drop = create_cnn(reg_type='l2', dropout_rate=0.5)
results['L2+Dropout'] = train_and_evaluate(model_l2_drop, "L2 + Dropout")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Training Base Model ---
Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.8273 - loss: 0.4726 - val_accuracy: 0.8755 - val_loss: 0.3496
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8939 - loss: 0.2881 - val_accuracy: 0.8920 - val_loss: 0.2939
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9107 - loss: 0.2425 - val_accuracy: 0.9009 - val_loss: 0.2768
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9222 - loss: 0.2107 - val_accuracy: 0.9039 - val_loss: 0.2817
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9317 - loss: 0.1841 - val_accuracy: 0.9091 - val_loss: 0.2649
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9393 - loss: 0.1624 - val_accuracy: 0.9076 - val_loss: 0.2722
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9475 - loss: 0.1407 - val_accuracy: 0.9061 - val_loss: 0.2953
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9535 - l

In [6]:
# --------------------- Final summary ---------------------
print("\n==================== SUMMARY ====================")
print(f"{'Model':<15} {'Train Acc':<12} {'Test Acc':<12}")
print("-" * 40)
for name, (train_acc, test_acc) in results.items():
    print(f"{name:<15} {train_acc:<12.4f} {test_acc:<12.4f}")


==================== SUMMARY ====================
Model           Train Acc    Test Acc    
----------------------------------------
Base            0.9626       0.9060      
L1              0.8372       0.8289      
L2              0.9100       0.9000      
Dropout         0.9517       0.9160      
L2+Dropout      0.9019       0.8989      
